# 07b - Control: TFG, then no guidance
Like `07`, but the last `k` steps have no guidance at all. If this matches TFG-then-OU, the OU guess adds nothing. Run after `07` (`k = 5` already exists), then `09` again.

Needs `ou_guidance/` and `resnet_cifar10.pt` in `My Drive/tfg_ou/`. Runtime: T4 GPU.
If the install cell asks to restart, restart and run from the top.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/tfg_ou'      # code, checkpoint and results

In [ ]:
# pinned: a newer diffusers clashes with transformers
!pip install -q diffusers==0.35.2 transformers==4.55.2 "huggingface-hub>=0.34,<1.0" torch-fidelity

In [ ]:
import sys
sys.path.insert(0, f'{ROOT}/ou_guidance')
import helpers
helpers.ROOT = ROOT
from helpers import run, search, show_results, diagnose

In [ ]:
LABELS = [8]        # CIFAR10 classes (8 = ship)
N = 100             # images per final run
N_SEARCH = 64       # images per search trial (as in the TFG repo)
OU_SWEEP = [5, 10, 20, 30, 40, 60]     # k: last steps without guidance (as in 07)

In [ ]:
for label in LABELS:
    tfg = search('network', 'tfg', label=label, n=N_SEARCH)       # done by 04_tfg: only reads the result
    for k in OU_SWEEP:
        run('network', label=label, n=N, **tfg, ou_steps=k, ou_rho=0.0, ou_mu=0.0)    # no guidance in the last k steps

## Results

In [ ]:
show_results(kinds=['unconditional', 'TFG', 'TFG-then-OU', 'TFG-then-unguided'])